In [ ]:
import duckdb

con = duckdb.connect("../data/warehouse.duckdb", read_only=True)

In [ ]:
con.sql("""
    SELECT p.player_name, t.team_short_name, p.position, p.price_m, p.selected_by_pct
    FROM stg_player_snapshots AS p
    JOIN stg_teams AS t ON t.team_id = p.team_id
    WHERE p.snapshot_date = (SELECT max(snapshot_date) FROM stg_player_snapshots)
    ORDER BY p.selected_by_pct DESC
    LIMIT 10
""").df()

In [ ]:
con.sql("SELECT * FROM mart_league_summary ORDER BY league_rank").df()

In [ ]:
con.sql("""
    WITH bounds AS (
        SELECT min(snapshot_date) AS first_day, max(snapshot_date) AS last_day
        FROM stg_player_snapshots
    )
    SELECT
        l.player_name,
        f.price_m AS first_price,
        l.price_m AS latest_price,
        round(l.price_m - f.price_m, 1) AS price_change
    FROM stg_player_snapshots AS f
    JOIN stg_player_snapshots AS l ON l.player_id = f.player_id
    JOIN bounds AS b ON f.snapshot_date = b.first_day AND l.snapshot_date = b.last_day
    WHERE l.price_m <> f.price_m
    ORDER BY price_change DESC
""").df()

In [ ]:
con.close()